# Public Policy Project 

## Workplace parking pricing and commuting mode

### By Nina Rindisbacher, Florian Loubies, Maxime Sheikhossen and Max Laurent

#### Under the supervision of Pr. Michaël Aklin

---
## 1. Context

A Swiss public employer operates two sites: a **main site in Lausanne (VD)**, and a **secondary site in Sion (VS)**.

Both sites charged staff **CHF 2 per day** for on-site parking in 2024. In 2025, the Lausanne fee increased to **CHF 4**, while the Sion fee **remained CHF 2**. No other mobility policy changed at either site.

All 1,000 employees were surveyed in both years. The employer must now choose its **2026 parking tariff**. Our group acts as a policy-analysis team advising both the **mobility manager** and the **staff council**.

The analysis is guided by three linked questions:
1. How many people stop driving when the price doubles?
2. What would happen at CHF 6 or CHF 8?
3. Who ends up paying?

**Objective:** build a transparent empirical argument from raw data to a defensible policy recommendation.

---
## 2. Data file

`parking_data.csv` contains 2,000 rows: 1,000 employees observed in 2024 and 2025.

| Variable | Type | Description |
|---|---|---|
| `employee_id` | string | Anonymised identifier, stable across years |
| `site` | string | Lausanne or Sion |
| `year` | int | 2024 or 2025 |
| `parking_fee_chf` | float | Daily fee in force at that site and in that year |
| `main_mode` | string | car, bus, or bike |
| `uses_car` | int | 1 if `main_mode == "car"`; 0 otherwise |
| `car_days_per_week` | float | Average days driven to work, from 0 to 5; 0 for non-car commuters |
| `home_commune` | string | Municipality of residence |
| `home_postcode` | int | Swiss postcode |
| `distance_km` | float | Road distance from home to the work site |
| `age` | int | Age |
| `gender` | string | F or M |
| `n_children` | int | Children under 16 in the household: 0, 1, or 2 (2 means "2 or more") |
| `fte` | float | Full-time equivalent contractual workload: 0.6, 0.8, or 1.0 |
| `gross_income_chf` | int | Annual gross income at the employee's observed workload |

**Assumption:** 46 working weeks per year.

> **Note:** The data contain deliberate complications. They must be identified and handled transparently (see the data-cleaning section).

---
## 3. Basic empirical framework

### 3.1 Two margins of car use
1. **Extensive margin:** whether the employee's main commuting mode is car (mode choice).
2. **Intensive margin:** how many days per week a car user drives (frequency).

### 3.2 Equation (1): mode choice (linear probability model)

$$\text{uses\_car} = a \cdot \log(\text{fee}) + \text{controls} + \log(\text{fee}) \times \text{interactions} \tag{1}$$

- Effects are reported in **probability units / percentage points**.
- Interactions: **transit index** (Part A), **income**, and **number of children**.
- The policy can be represented either through $\log(\text{fee})$ or through the **Lausanne × 2025** interaction ($L_i \times \text{Post}_t$) in a site-and-year comparison model. With the same sample, controls and interactions, these are equivalent parameterisations of the same policy variation, but the coefficients have different units.

### 3.3 Equation (2): frequency (car users only)

$$\log(\text{car\_days\_week}) = b \cdot \log(\text{fee}) + \text{controls} + \log(\text{fee}) \times \text{interactions} \tag{2}$$

- Because both the outcome and the fee are logged, $b$ is **directly an elasticity**.
- Interactions: **transit index** (Part A), **income**, and **number of children**.

---
# Part A: Public-transport accessibility by commune

**Goal:** for each home commune, measure how realistic public transport (PT) is as an alternative to the car for arriving at work around 08:00.

**Input:** `parking_data.csv`
**Outputs:**
- `transit_access.csv` (one row per commune)
- `table_A1.csv` (Table A1)
- `transit_cache.json` (archive of API responses: re-running gives identical results)

**Sources:**
- transport.opendata.ch (official SBB timetables)
- api3.geo.admin.ch (swisstopo: communes and the federal Buildings and Dwellings Register, RegBL)
- Google Maps (car travel times)

Run the cells in order (*Run All*).

## A.0 Libraries

In [1]:
import json, math, os, re, time, unicodedata
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from threading import RLock

import numpy as np
import pandas as pd
import requests

## A.0b Parameters

All methodological choices are set in the cell below.

**Timetable**
- **Reference date:** Thursday 10 Dec 2026, the last working day of the 2026 timetable. The API does not provide past timetables (2024–2025).
- **Destination:** the main railway station of the site (Lausanne or Sion), since the exact addresses of the sites are not known.
- **Excluded services:** lines introduced after the 2024–2025 survey (line 9 extended to Renens/Crissier, line 54, tram T1).

**Choice of the departure stop for each commune**
1. *Dwellings:* all residential buildings of the commune (federal register RegBL), each weighted by its number of dwellings.
2. *Candidate stops:* stops close to the mean dwelling location and to the centre of each quarter of the commune (so that a station on the edge is not missed), plus the commune's main stop (`MAIN_STOP`). Only stops with at least one departure between 06:00 and 09:00 are kept.
3. *Mean total time* of each stop = average walk from all dwellings to the stop + fastest PT trip from the stop to the site's station, arriving between 07:30 and 08:00.
4. *Selected stop:* the one with the lowest mean total time. If several stops are within `TIE_MIN` = 1 minute of this minimum (the timetable is accurate to the minute), keep the one with the shortest average walk.

**Walking and car**
- **Walking:** straight-line distance × `WALK_DETOUR` = 1.3 (street detour), at `WALK_M_PER_MIN` = 75 m/min (4.5 km/h).
- **Car:** Google Maps time (`CAR_TIMES`, from the mean dwelling location to the station, arriving 08:00 on a Thursday). We take the middle of the displayed range and add `CAR_TERMINAL_MIN` = 5 min to park and walk to the site.

In [2]:
REF_DATE = "2026-12-10"     # Thursday, last working day of the 2026 timetable before the
                            # 13.12.2026 change (no tram T1 yet)
WALK_DETOUR = 1.3           # walking path = 1.3 x straight-line distance
WALK_M_PER_MIN = 75         # 4.5 km/h
CAR_TERMINAL_MIN = 5        # parking + walking to the site
TIE_MIN = 1.0               # stops within 1 min of each other = tie -> shortest walk

SITE_STOP = {"Lausanne": "Lausanne", "Sion": "Sion"}           # destination = main station
SITE_COORD = {"Lausanne": (46.5167, 6.6291), "Sion": (46.2274, 7.3592)}

# Main stop of each locality (always added to the candidate stops)
MAIN_STOP = {
    "BOTTENS": "Bottens VD", "BUSSIGNY": "Bussigny", "CHAVANNES PRES RENENS": "Chavannes-près-Renens",
    "CHESEAUX SUR LAUSANNE": "Cheseaux", "CORCELLES LE JORAT": "Corcelles-le-Jorat",
    "CRISSIER": "Crissier, centre", "CUGY": "Cugy VD", "ECUBLENS": "Ecublens VD",
    "EPALINGES": "Epalinges, Croisettes", "FROIDEVILLE": "Froideville", "LAUSANNE": "Lausanne, Chauderon",
    "LE MONT SUR LAUSANNE": "Le Mont-sur-Lausanne", "LUTRY": "Lutry", "MEZIERES": "Mézières VD",
    "MONTPREVEYRES": "Montpreveyres", "MORGES": "Morges", "PRILLY": "Prilly", "PULLY": "Pully",
    "RENENS": "Renens VD", "ROMANEL SUR LAUSANNE": "Romanel-sur-Lausanne", "ROPRAZ": "Ropraz",
    "SAVIGNY": "Savigny VD", "SERVION": "Servion", "VULLIENS": "Vulliens", "ARBAZ": "Arbaz",
    "ARDON": "Ardon", "AYENT": "Ayent", "BRAMOIS": "Bramois", "CONTHEY": "Conthey",
    "EVOLENE": "Evolène", "GRIMISUAT": "Grimisuat", "HEREMENCE": "Hérémence", "NENDAZ": "Basse-Nendaz",
    "SAINT MARTIN": "Saint-Martin VS", "SALINS": "Salins VS", "SAVIESE": "Savièse",
    "SION": "Sion, Platta", "VETROZ": "Vétroz", "VEX": "Vex",
}
NPA_ONLY = {"LAUSANNE", "BRAMOIS", "SALINS"}  # dwellings selected by postcode (neighbourhood / former village)

# Services introduced AFTER the 2024-2025 survey -> excluded
#   line 9 extended to Renens/Crissier (08.2026), line 54 (12.2025), tram T1
NEW_SERVICES = {("B", "9"): ["Crissier", "Renens"], ("B", "54"): None,
                ("T", "1"): None, ("T", "T1"): None}

# Google Maps car times (min), from the mean dwelling location to the site's station,
# arriving 08:00 on a Thursday: (lower, upper) bound of the displayed range
CAR_TIMES = {
    "ARBAZ": (20, 30), "ARDON": (14, 22), "AYENT": (20, 20), "BOTTENS": (24, 45), "BRAMOIS": (9, 12),
    "BUSSIGNY": (14, 24), "CHAVANNES PRES RENENS": (12, 20), "CHESEAUX SUR LAUSANNE": (18, 35),
    "CONTHEY": (8, 14), "CORCELLES LE JORAT": (26, 50), "CRISSIER": (16, 28), "CUGY": (22, 40),
    "ECUBLENS": (14, 24), "EPALINGES": (14, 35), "EVOLENE": (35, 35), "FROIDEVILLE": (24, 45),
    "GRIMISUAT": (12, 20), "HEREMENCE": (22, 30), "LAUSANNE": (8, 16), "LE MONT SUR LAUSANNE": (14, 28),
    "LUTRY": (12, 22), "MEZIERES": (26, 45), "MONTPREVEYRES": (24, 45), "MORGES": (16, 24),
    "NENDAZ": (26, 30), "PRILLY": (9, 16), "PULLY": (7, 16), "RENENS": (12, 22),
    "ROMANEL SUR LAUSANNE": (12, 24), "ROPRAZ": (26, 50), "SAINT MARTIN": (31, 31), "SALINS": (14, 14),
    "SAVIESE": (16, 24), "SAVIGNY": (18, 35), "SERVION": (26, 45), "SION": (4, 4), "VETROZ": (12, 18),
    "VEX": (14, 18), "VULLIENS": (30, 55),
}

## A.1 Raw data: inspection and cleaning

We first look at the data as provided (A.1a to A.1e) to spot missing data, impossible values and inconsistencies. We then correct the communes (A.1f) and the remaining issues (A.1g), and aggregate by commune (A.1h). The file `parking_data.csv` itself is never modified.

### A.1a Overview

In [3]:
raw = pd.read_csv("parking_data.csv")
print(raw.shape)
print("Employees:", raw["employee_id"].nunique(), "| years:", sorted(raw["year"].unique()))
print("Rows per employee:", raw.groupby("employee_id").size().value_counts().to_dict())
print("Employees per site:", raw.drop_duplicates("employee_id")["site"].value_counts().to_dict())
raw.head()

(2000, 15)
Employees: 1000 | years: [np.int64(2024), np.int64(2025)]
Rows per employee: {2: 1000}
Employees per site: {'Lausanne': 800, 'Sion': 200}


,employee_id,site,year,parking_fee_chf,main_mode,uses_car,car_days_per_week,home_commune,home_postcode,distance_km,age,gender,n_children,fte,gross_income_chf
0,E1000,Lausanne,2024,2.0,car,1,2.2,Lausanne,1004,2.9,23,M,0,0.8,59300.0
1,E1000,Lausanne,2025,4.0,car,1,2.0,Lausanne,1004,2.9,23,M,0,0.8,59300.0
2,E1001,Lausanne,2024,2.0,car,1,3.5,Le Mont-sur-Lausanne,1052,5.6,51,F,1,1.0,124300.0
3,E1001,Lausanne,2025,4.0,car,1,7.4,Le Mont-sur-Lausanne,1052,5.6,51,F,1,1.0,124300.0
4,E1002,Lausanne,2024,2.0,car,1,2.6,Chavannes-pres-Renens,1022,7.7,39,M,0,1.0,72800.0


### A.1b Missing data and duplicates

In [4]:
print("Missing values per column:")
print(raw.isna().sum()[raw.isna().sum() > 0])
print("\nDuplicates (same employee, same year):", raw.duplicated(["employee_id", "year"]).sum())
miss = raw[raw["gross_income_chf"].isna()]
n_years = miss.groupby("employee_id").size()
print("Missing income:", miss["employee_id"].nunique(), "employees, of which",
      (n_years == 1).sum(), "for one year only and", (n_years == 2).sum(), "for both years")

Missing values per column:
gross_income_chf    110
dtype: int64

Duplicates (same employee, same year): 0
Missing income: 106 employees, of which 102 for one year only and 4 for both years


### A.1c Consistency between 2024 and 2025

These characteristics should not change from one year to the next for a given employee.

In [5]:
for col in ["site", "home_commune", "home_postcode", "distance_km", "age",
            "n_children", "fte", "gross_income_chf"]:
    n = (raw.groupby("employee_id")[col].nunique() > 1).sum()
    print(f"{col:<18} changes for {n} employees")

site               changes for 0 employees
home_commune       changes for 45 employees
home_postcode      changes for 0 employees
distance_km        changes for 0 employees
age                changes for 0 employees
n_children         changes for 0 employees
fte                changes for 0 employees
gross_income_chf   changes for 0 employees


### A.1d Value ranges and logical consistency

We compare the data with the values allowed by the variable descriptions (e.g. `car_days_per_week` between 0 and 5).

In [6]:
print(raw.describe().T[["min", "max", "mean"]])
print("\ncar_days_per_week > 5:")
print(raw.loc[raw["car_days_per_week"] > 5, ["employee_id", "year", "main_mode", "car_days_per_week"]])
print("\nuses_car inconsistent with main_mode:", ((raw["main_mode"] == "car") != (raw["uses_car"] == 1)).sum())
print("Non-car commuters with car days:", ((raw["uses_car"] == 0) & (raw["car_days_per_week"] > 0)).sum())
print("\nModes:", raw["main_mode"].value_counts().to_dict())
print("\nParking fee by site and year:")
print(raw.groupby(["site", "year"])["parking_fee_chf"].unique())

                       min       max          mean
year                2024.0    2025.0   2024.500000
parking_fee_chf        2.0       4.0      2.800000
uses_car               0.0       1.0      0.511000
car_days_per_week      0.0       7.4      1.815650
home_postcode       1004.0    1997.0   1224.999000
distance_km            1.3      37.5      8.343900
age                   22.0      64.0     42.567000
n_children             0.0       2.0      0.748000
fte                    0.6       1.0      0.962600
gross_income_chf   24300.0  222500.0  94784.338624

car_days_per_week > 5:
  employee_id  year main_mode  car_days_per_week
3       E1001  2025       car                7.4

uses_car inconsistent with main_mode: 0
Non-car commuters with car days: 0

Modes: {'car': 1022, 'bus': 705, 'bike': 273}

Parking fee by site and year:
site      year
Lausanne  2024    [2.0]
          2025    [4.0]
Sion      2024    [2.0]
          2025    [2.0]
Name: parking_fee_chf, dtype: object


### A.1e Communes and postcodes

Commune names are written in several ways (accents, hyphens, capitalisation). We harmonise them to see the variants, then check that each commune has only one postcode and that it is the correct one.

In [7]:
def normalize(name):
    """Upper case, no accents or punctuation: 'Chavannes-près-Renens' -> 'CHAVANNES PRES RENENS'."""
    name = unicodedata.normalize("NFKD", str(name).upper().strip())
    name = "".join(ch for ch in name if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9]", " ", name)).strip()

check = raw.assign(commune_clean=raw["home_commune"].apply(normalize))
print("Distinct spellings:", raw["home_commune"].nunique(), "-> communes after harmonisation:",
      check["commune_clean"].nunique())
pd.set_option("display.max_rows", 100)
check.groupby("commune_clean").agg(
    site=("site", "unique"),
    variants=("home_commune", lambda s: sorted(s.unique())),
    postcode=("home_postcode", "unique"),
    employees=("employee_id", "nunique"),
)

Distinct spellings: 62 -> communes after harmonisation: 39


,site,variants,postcode,employees
commune_clean,,,,
ARBAZ,[Sion],[Arbaz],[1974],6
ARDON,[Sion],"[ARDON, Ardon]",[1957],11
AYENT,[Sion],"[AYENT, Ayent]",[1966],15
BOTTENS,[Lausanne],"[BOTTENS, Bottens]",[1041],15
BRAMOIS,[Sion],[Bramois],[1967],16
BUSSIGNY,[Lausanne],"[BUSSIGNY, Bussigny]",[1030],43
CHAVANNES PRES RENENS,[Lausanne],"[CHAVANNES PRES RENENS, Chavannes-pres-Renens]",[1022],20
CHESEAUX SUR LAUSANNE,[Lausanne],[Cheseaux-sur-Lausanne],[1033],21
CONTHEY,[Sion],[Conthey],[1964],20


**What the inspection shows:**
- 62 spellings for 39 communes (accents, hyphens, capitalisation) → harmonisation needed.
- **Savièse** is written "Savieze" in all rows (spelling mistake).
- **Salins:** all 8 employees have postcode **1994**, which is the postcode of Aproz (a part of the commune of Nendaz). The postcode of Salins is 1991.
- Each commune has a single postcode. Site, postcode, distance, age, children, FTE and income never change between 2024 and 2025.
- `home_commune` changes between 2024 and 2025 for 45 employees, but **only in spelling** (e.g. "FROIDEVILLE" / "Froideville"). After harmonisation, no employee changes commune (checked in A.1f).
- **Income** is the only variable with missing values (110 rows, 106 employees, 102 of them for one year only).
- **Impossible value:** E1001 reports 7.4 car days per week in 2025 (maximum is 5).
- `uses_car` always matches `main_mode`, and the fees match the project description (Lausanne: CHF 2 → 4; Sion: CHF 2).

### A.1f Correcting the communes

- Names harmonised (`commune_clean`).
- "Savieze" → **SAVIESE**.
- Salins: we trust the commune name → postcode corrected from 1994 to **1991**.

The original values are kept (`commune_clean_raw`, `home_postcode_raw`).

In [8]:
raw["commune_clean"] = raw["home_commune"].apply(normalize)
raw["commune_clean_raw"] = raw["commune_clean"]
raw["home_postcode_raw"] = raw["home_postcode"]

raw["commune_clean"] = raw["commune_clean"].replace({"SAVIEZE": "SAVIESE"})   # spelling mistake
raw.loc[raw["commune_clean"] == "SALINS", "home_postcode"] = 1991              # postcode 1994 = Aproz

print("Communes:", raw["commune_clean"].nunique())
print("Employees changing commune between 2024 and 2025 (after harmonisation):",
      (raw.groupby("employee_id")["commune_clean"].nunique() > 1).sum())
raw.loc[raw["commune_clean"].isin(["SAVIESE", "SALINS"]),
        ["commune_clean_raw", "commune_clean", "home_postcode_raw", "home_postcode"]].drop_duplicates()

Communes: 39
Employees changing commune between 2024 and 2025 (after harmonisation): 0


,commune_clean_raw,commune_clean,home_postcode_raw,home_postcode
1632,SAVIEZE,SAVIESE,1965,1965
1660,SALINS,SALINS,1994,1991


### A.1g Other corrections

- **Income missing for one year only:** we use the value from the other year (income is constant within employee, see A.1c). Employees with no income in either year remain missing. This affects 4 employees (0.4%, i.e. 8 rows). They will have to be excluded from income-based analyses, or imputed with the median of similar profiles (FTE, age, work site).
- **Car days > 5** (impossible): capped at 5.

Original values are kept: `income_imputed` (flag for completed income), `car_days_raw`. These corrections do not affect the Part A results (which only use communes), but they are used in Parts B to E.

In [9]:
raw["income_imputed"] = raw["gross_income_chf"].isna()
raw["gross_income_chf"] = raw.groupby("employee_id")["gross_income_chf"].transform(lambda s: s.fillna(s.mean()))
raw["income_imputed"] &= raw["gross_income_chf"].notna()
print("Incomes completed:", raw["income_imputed"].sum(),
      "| still missing:", raw["gross_income_chf"].isna().sum(), "rows")

raw["car_days_raw"] = raw["car_days_per_week"]
raw["car_days_per_week"] = raw["car_days_per_week"].clip(upper=5)
print("Car days capped at 5:", (raw["car_days_raw"] > 5).sum(), "row(s)")

Incomes completed: 102 | still missing: 8 rows
Car days capped at 5: 1 row(s)


### A.1h One row per commune

In [10]:
communes = (raw.groupby(["commune_clean", "site"])
               .agg(n_employees=("employee_id", "nunique"),
                    postcode=("home_postcode", "first"),
                    mean_distance_km=("distance_km", "mean"))
               .reset_index())
assert len(communes) == 39 and communes.n_employees.sum() == 1000
communes

,commune_clean,site,n_employees,postcode,mean_distance_km
0,ARBAZ,Sion,6,1974,11.483333
1,ARDON,Sion,11,1957,13.809091
2,AYENT,Sion,15,1966,12.773333
3,BOTTENS,Lausanne,15,1041,14.440000
4,BRAMOIS,Sion,16,1967,5.318750
5,BUSSIGNY,Lausanne,43,1030,9.788372
6,CHAVANNES PRES RENENS,Lausanne,20,1022,7.110000
7,CHESEAUX SUR LAUSANNE,Lausanne,21,1033,8.547619
8,CONTHEY,Sion,20,1964,7.225000
9,CORCELLES LE JORAT,Lausanne,13,1082,19.423077


## A.2 API access

Each response is saved in `transit_cache.json`: a request is made only once, and re-running the notebook gives exactly the same results.
Connections that use a service introduced after 2025 are discarded.

In [11]:
CACHE_FILE = "transit_cache.json"
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}
lock = RLock()


def save_cache():
    with lock:
        with open(CACHE_FILE + ".tmp", "w") as f:
            json.dump(cache, f)
        os.replace(CACHE_FILE + ".tmp", CACHE_FILE)


def get(url, params, transform=None):
    key = url + "?" + json.dumps(params, sort_keys=True)
    if key in cache:
        return cache[key]
    for attempt in range(4):                       # retry if the API is overloaded
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code != 429:
                break
        except requests.ConnectionError:
            pass
        time.sleep(5 * (attempt + 1))
    r.raise_for_status()
    data = transform(r.json()) if transform else r.json()
    with lock:
        cache[key] = data
        if len(cache) % 25 == 0:
            save_cache()
    time.sleep(0.2)
    return data


OPENDATA = "https://transport.opendata.ch/v1/"
GEO = "https://api3.geo.admin.ch/rest/services/api/"


def ts(s):                                         # '2026-12-10T07:52:00+0100' -> datetime
    return datetime.strptime(s[:19], "%Y-%m-%dT%H:%M:%S")


def at(hhmm):                                      # '08:00' -> datetime on the reference day
    return datetime.strptime(f"{REF_DATE} {hhmm}", "%Y-%m-%d %H:%M")


def km(lat1, lon1, lat2, lon2):                    # straight-line (haversine) distance
    p = math.pi / 180
    a = (math.sin((lat2 - lat1) * p / 2) ** 2
         + math.cos(lat1 * p) * math.cos(lat2 * p) * math.sin((lon2 - lon1) * p / 2) ** 2)
    return 12742 * math.asin(math.sqrt(a))


def is_new_service(category, number, boarding_stop):
    prefixes = NEW_SERVICES.get((category, str(number)), False)
    return prefixes is None or (prefixes and any(boarding_stop.startswith(p) for p in prefixes))


def connections(frm, to, hhmm, arrival, limit):
    """PT connections, excluding those that use a service introduced after 2025."""
    res = get(OPENDATA + "connections", {"from": frm, "to": to, "date": REF_DATE, "time": hhmm,
                                         "isArrivalTime": int(arrival), "limit": limit, "page": 0})
    ok = [c for c in res.get("connections", [])
          if not any(s.get("journey") and is_new_service(s["journey"].get("category"),
                                                         s["journey"].get("number"),
                                                         s["departure"]["station"]["name"] or "")
                     for s in c.get("sections", []))]
    return ok, res.get("connections", [])

## A.3 The three PT indicators for a stop

1. **08:00 trip:** the fastest trip arriving between 07:30 and 08:00 (otherwise the last one before 08:00).
2. **Departures 06:00–09:00:** number of distinct departures to the site between 06:00 and 09:00.
3. **First arrival:** time of the first possible morning arrival at the site.

In [12]:
def trip_0800(stop, site):
    """Fastest trip arriving between 07:30 and 08:00 (otherwise the last one before 08:00)."""
    cons, _ = connections(stop, SITE_STOP[site], "08:00", arrival=True, limit=6)
    cons = [c for c in cons if ts(c["to"]["arrival"]) <= at("08:00")]
    pool = ([c for c in cons if ts(c["to"]["arrival"]) >= at("07:30")]
            or sorted(cons, key=lambda c: ts(c["to"]["arrival"]))[-1:])
    if not pool:
        return None
    return min(pool, key=lambda c: ts(c["to"]["arrival"]) - ts(c["from"]["departure"]))


def trip_minutes(c):
    return (ts(c["to"]["arrival"]) - ts(c["from"]["departure"])).total_seconds() / 60


def departures_6_9(stop, site):
    """Distinct departures 06:00-09:00 to the site (the API returns 16 connections per request)."""
    deps, hhmm = set(), "06:00"
    for _ in range(30):
        ok, allc = connections(stop, SITE_STOP[site], hhmm, arrival=False, limit=16)
        if not allc:
            break
        deps |= {ts(c["from"]["departure"]) for c in ok
                 if at("06:00") <= ts(c["from"]["departure"]) < at("09:00")}
        last = max(ts(c["from"]["departure"]) for c in allc)
        nxt = (last + timedelta(minutes=1)).strftime("%H:%M")
        if last >= at("09:00") or nxt <= hhmm:
            break
        hhmm = nxt
    return len(deps)


def first_arrival(stop, site):
    ok, _ = connections(stop, SITE_STOP[site], "04:00", arrival=False, limit=4)
    arr = [ts(c["to"]["arrival"]) for c in ok if ts(c["to"]["arrival"]).date() == at("04:00").date()]
    return min(arr).strftime("%H:%M") if arr else None


def served_6_9(stop_id, stop_name):
    """At least one departure (any direction) between 06:00 and 09:00 at this stop?"""
    r = get(OPENDATA + "stationboard", {"id": stop_id, "datetime": f"{REF_DATE} 06:00", "limit": 200})
    return any(d.get("stop", {}).get("departure")
               and at("06:00") <= ts(d["stop"]["departure"]) < at("09:00")
               and not is_new_service(d.get("category"), d.get("number"), stop_name)
               for d in r.get("stationboard", []))

## A.4 Dwellings and selected stop for a commune

1. **Dwellings:** all residential buildings of the locality (federal register RegBL), weighted by their number of dwellings. For Lausanne, only the employees' postcode (1004) is used.
2. **Candidate stops:** stops close to the mean dwelling location and to the centre of each quarter of the locality, plus the main stop.
3. **Selected stop:** among the stops served between 06:00 and 09:00, the one that minimises **average walk from all dwellings + 08:00 PT trip**. A tie (< 1 min) → shortest walk.

In [13]:
def commune_centre(commune, site, road_km):
    """Centre of the commune (swisstopo). Rejects homonyms that are too far from the site."""
    road_km = max(road_km, 10) if commune == site.upper() else road_km
    for layer in ("gg25", "gazetteer"):            # gg25 = communes ; gazetteer = villages
        r = get(GEO + "SearchServer", {"searchText": commune.title(), "type": "locations",
                                       "origins": layer, "sr": "4326"})
        for res in r.get("results", []):
            a = res["attrs"]
            label = re.sub(r"<[^>]+>", "", a.get("label", ""))
            if commune in normalize(label) and km(a["lat"], a["lon"], *SITE_COORD[site]) <= road_km + 3:
                return a["lat"], a["lon"], a.get("featureId") if layer == "gg25" else None
    return None


def compact(data):                                 # keep only 5 columns of the register
    keep = ("egid", "ganzwhg", "gkat", "gstat", "plz_plz6")
    return {"results": [{"featureId": f.get("featureId"),
                         "attributes": {k: f.get("attributes", {}).get(k) for k in keep},
                         "geometry": {"x": f.get("geometry", {}).get("x"),
                                      "y": f.get("geometry", {}).get("y")}}
                        for f in data.get("results", [])]}


def dwellings(search_value, field):
    """Residential buildings (RegBL): list of (lat, lon, number of dwellings, postcode). ~200 rows per page."""
    rows, seen, offset = [], set(), 0
    while True:
        page = get(GEO + "MapServer/find",
                   {"layer": "ch.bfs.gebaeude_wohnungs_register", "searchText": str(search_value),
                    "searchField": field, "returnGeometry": "true", "sr": "4326",
                    "contains": "false", "offset": offset}, transform=compact)["results"]
        new = [f for f in page if f["featureId"] not in seen]
        if not new:
            break
        seen |= {f["featureId"] for f in new}
        rows += new
        offset += 200
        if len(page) < 200:
            break
    out, buildings = [], set()
    for f in rows:                                 # one row per dwelling -> one per building
        a, g = f["attributes"], f["geometry"]
        if g["x"] is None or a["egid"] in buildings or str(a.get("gstat")) not in ("1004", "None"):
            continue
        buildings.add(a["egid"])
        w = float(a.get("ganzwhg") or 0) or (1 if str(a.get("gkat")) in ("1020", "1030", "1040") else 0)
        if w > 0:
            out.append((g["y"], g["x"], w, str(a.get("plz_plz6", ""))[:4]))
    return out


def stops_near(lat, lon):
    r = get(OPENDATA + "locations", {"x": lat, "y": lon, "type": "station"})
    found = [s for s in r.get("stations", []) if s.get("id") and (s.get("coordinate") or {}).get("x")]
    found.sort(key=lambda s: s.get("distance") or 0)               # nearest to farthest
    return [(s["id"], s["name"], s["coordinate"]["x"], s["coordinate"]["y"]) for s in found]


def wmean(values, weights):
    return sum(v * w for v, w in zip(values, weights)) / sum(weights)


def analyse_commune(commune, site, npa, road_km):
    # --- dwellings of the locality ---
    if commune in NPA_ONLY:                        # buildings of the employees' postcode
        homes = dwellings(npa, "dplz4")            # (Lausanne 1004, Bramois 1967, Salins 1991)
    else:
        centre = commune_centre(commune, site, road_km)
        homes = dwellings(centre[2], "ggdenr")
        same_npa = [h for h in homes if h[3] == str(npa)]
        homes = same_npa if len(same_npa) >= 10 else homes
    lat, lon, w = [h[0] for h in homes], [h[1] for h in homes], [h[2] for h in homes]
    clat, clon = wmean(lat, w), wmean(lon, w)

    # --- candidate stops: around the centre, the centre of each quarter, + main stop ---
    cands = {s[0]: s for s in stops_near(clat, clon)}
    mlat, mlon = sorted(lat)[len(lat) // 2], sorted(lon)[len(lon) // 2]
    for north in (True, False):
        for east in (True, False):
            q = [h for h in homes if (h[0] >= mlat) == north and (h[1] >= mlon) == east]
            if q:
                qlat, qlon = wmean([h[0] for h in q], [h[2] for h in q]), wmean([h[1] for h in q], [h[2] for h in q])
                for s in [s for s in stops_near(qlat, qlon) if s[0] not in cands][:5]:
                    cands[s[0]] = s
        # + stops near the outermost dwellings (N, S, E, W): stations on the commune boundary
    for h in (max(homes, key=lambda h: h[0]), min(homes, key=lambda h: h[0]),
              max(homes, key=lambda h: h[1]), min(homes, key=lambda h: h[1])):
        for s in [s for s in stops_near(h[0], h[1]) if s[0] not in cands][:5]:
            cands[s[0]] = s
    for name in [MAIN_STOP[commune]]:
        found = [(s["id"], s["name"], s["coordinate"]["x"], s["coordinate"]["y"])
                 for s in get(OPENDATA + "locations", {"query": name, "type": "station"}).get("stations", [])
                 if s.get("id") and (s.get("coordinate") or {}).get("x")]
        if found:
            cands.setdefault(found[0][0], found[0])
    cands = list(cands.values())

    # --- served stops and their trip to the site (4 parallel requests) ---
    with ThreadPoolExecutor(4) as ex:
        served = [s for s, ok in zip(cands, ex.map(lambda s: served_6_9(s[0], s[1]), cands)) if ok]
        trips = dict(zip([s[0] for s in served], ex.map(lambda s: trip_0800(s[1], site), served)))
    served = [s for s in served if trips[s[0]]]

    # --- selected stop: lowest (average walk + trip); tie -> shortest walk ---
    def walk(s):
        return wmean([km(h[0], h[1], s[2], s[3]) * 1000 * WALK_DETOUR / WALK_M_PER_MIN for h in homes], w)

    total = {s[0]: walk(s) + trip_minutes(trips[s[0]]) for s in served}
    best = min([s for s in served if total[s[0]] < min(total.values()) + TIE_MIN], key=walk)

    # share of dwellings for which this stop would also be the best individual choice
    def own_best(h):
        return min(served, key=lambda s: km(h[0], h[1], s[2], s[3]) * 1000 * WALK_DETOUR / WALK_M_PER_MIN
                   + trip_minutes(trips[s[0]]))[0]
    share = sum(h[2] for h in homes if own_best(h) == best[0]) / sum(w)

    trip = trips[best[0]]
    return {
        "origin_stop": best[1],
        "n_dwellings": int(sum(w)),
        "share_dwellings_using_stop": round(share, 2),
        "dwellings_centre_latlon": f"{clat:.5f}, {clon:.5f}",
        "walk_to_origin_stop_min": round(walk(best), 1),
        "pt_stop_to_stop_min": trip_minutes(trip),
        "pt_transfers": trip.get("transfers"),
        "pt_total_via_origin_min": round(walk(best) + trip_minutes(trip), 1),
        "n_departures_6_9": departures_6_9(best[1], site),
        "arrival_before_0800": ts(trip["to"]["arrival"]).strftime("%H:%M"),
        "first_arrival": first_arrival(best[1], site),
    }

## A.5 Computation for the 39 communes

The switch `RUN_API` controls the source of the commune-level measure:
- `RUN_API = False` (default): load the precomputed `transit_access.csv`. Instant.
- `RUN_API = True`: recompute everything from the APIs (needs internet; about 15 to 45 minutes on a first run, a few seconds if `transit_cache.json` is present).

The file `transit_access.csv` in the submission was produced with `RUN_API = True`.

In [19]:
RUN_API = True   # True = recompute from the APIs (internet needed); False = load transit_access.csv

### Sanity check: Chavannes-près-Renens

The selected stop for a commune is not always the nearest station. This check shows that Renens station is among the candidate stops for Chavannes-près-Renens, and compares it directly with the stop that was selected (Concorde). The two options are close in total time (about 21 min), which is why the tie rule matters.

In [20]:
if RUN_API:
    # Chavannes: is Renens station a candidate, and how does it compare with Concorde?
    commune, site = "CHAVANNES PRES RENENS", "Lausanne"
    r = communes[communes.commune_clean == commune].iloc[0]
    centre = commune_centre(commune, site, r.mean_distance_km)
    homes = [h for h in dwellings(centre[2], "ggdenr") if h[3] == str(r.postcode)]
    w = [h[2] for h in homes]

    edge = set()
    for h in (max(homes, key=lambda h: h[0]), min(homes, key=lambda h: h[0]),
              max(homes, key=lambda h: h[1]), min(homes, key=lambda h: h[1])):
        edge |= {s[1] for s in stops_near(h[0], h[1])[:5]}
    print("Renens station found via the edge dwellings:", "Renens VD" in edge)

    for name in ["Chavannes-R., Concorde", "Renens VD"]:
        s = get(OPENDATA + "locations", {"query": name, "type": "station"})["stations"][0]
        walking = wmean([km(h[0], h[1], s["coordinate"]["x"], s["coordinate"]["y"]) * 1000
                         * WALK_DETOUR / WALK_M_PER_MIN for h in homes], w)
        trip_time = trip_minutes(trip_0800(s["name"], site))
        print(f"{s['name']:<25} walk {walking:5.1f} + trip {trip_time:4.0f} = {walking + trip_time:5.1f} min")
else:
    print("Skipped (RUN_API = False): this check queries the live APIs.")

Renens station found via the edge dwellings: True
Chavannes-R., Concorde    walk   9.6 + trip   12 =  21.6 min
Renens VD                 walk  15.8 + trip    5 =  20.8 min


### For the 39 communes

In [21]:
if RUN_API:
    rows = []
    try:
        for i, r in communes.iterrows():
            print(f"[{i + 1}/39] {r.commune_clean}", end=" ", flush=True)
            res = analyse_commune(r.commune_clean, r.site, r.postcode, r.mean_distance_km)
            print(f"-> {res['origin_stop']} ({res['pt_total_via_origin_min']} min)")
            rows.append({**r.to_dict(), **res})
    finally:
        save_cache()
    ta = pd.DataFrame(rows)
else:
    ta = pd.read_csv("transit_access.csv")

ta.head()

[1/39] ARBAZ -> Arbaz, poste (40.2 min)
[2/39] ARDON -> Ardon (23.2 min)
[3/39] AYENT -> Botyre (Ayent), Le Creux (46.0 min)
[4/39] BOTTENS -> Bottens, croisée (42.0 min)
[5/39] BRAMOIS -> Bramois, La Borgne (20.9 min)
[6/39] BUSSIGNY -> Bussigny, gare (20.6 min)
[7/39] CHAVANNES PRES RENENS -> Chavannes-R., Concorde (21.6 min)
[8/39] CHESEAUX SUR LAUSANNE -> Cheseaux-sur-L., gare (30.4 min)
[9/39] CONTHEY -> Châteauneuf-Conthey, gare (21.1 min)
[10/39] CORCELLES LE JORAT -> Corcelles-le-Jorat, village (55.9 min)
[11/39] CRISSIER -> Crissier, Marcolet (23.3 min)
[12/39] CUGY -> Cugy VD, village (36.4 min)
[13/39] ECUBLENS -> Chavannes-R., Crochy (24.7 min)
[14/39] EPALINGES -> Epalinges, Croisettes (31.8 min)
[15/39] EVOLENE -> Evolène, Village (49.7 min)
[16/39] FROIDEVILLE -> Froideville, croisée (49.0 min)
[17/39] GRIMISUAT -> Grimisuat, Les Fermes (32.8 min)
[18/39] HEREMENCE -> Euseigne, Pont de Sauterot (63.1 min)
[19/39] LAUSANNE -> Lausanne, St-Paul (19.8 min)
[20/39] LE MONT S

,commune_clean,site,n_employees,postcode,mean_distance_km,origin_stop,n_dwellings,share_dwellings_using_stop,dwellings_centre_latlon,walk_to_origin_stop_min,pt_stop_to_stop_min,pt_transfers,pt_total_via_origin_min,n_departures_6_9,arrival_before_0800,first_arrival
0,ARBAZ,Sion,6,1974,11.483333,"Arbaz, poste",976,0.23,"46.27759, 7.38421",12.2,28.0,1,40.2,4,07:50,06:47
1,ARDON,Sion,11,1957,13.809091,Ardon,1808,0.47,"46.20972, 7.25950",16.2,7.0,0,23.2,7,07:40,04:45
2,AYENT,Sion,15,1966,12.773333,"Botyre (Ayent), Le Creux",1973,0.05,"46.27920, 7.40956",19.0,27.0,0,46.0,6,07:48,06:30
3,BOTTENS,Lausanne,15,1041,14.440000,"Bottens, croisée",607,0.96,"46.61618, 6.66106",6.0,36.0,1,42.0,11,07:34,06:04
4,BRAMOIS,Sion,16,1967,5.318750,"Bramois, La Borgne",2045,0.24,"46.23141, 7.40070",10.9,10.0,0,20.9,19,07:51,06:05


### Car time and PT/car ratio

Car time = middle of the Google range + 5 min. Ratio = total PT time (walk + trip) ÷ car time. A ratio < 1 means PT is faster than the car.

In [22]:
# Car time and PT / car ratio
ta["car_min_low"] = ta.commune_clean.map(lambda c: CAR_TIMES[c][0])
ta["car_min_high"] = ta.commune_clean.map(lambda c: CAR_TIMES[c][1])
ta["car_door_mid_min"] = (ta.car_min_low + ta.car_min_high) / 2 + CAR_TERMINAL_MIN
ta["ratio_pt_car_mid"] = (ta.pt_total_via_origin_min / ta.car_door_mid_min).round(2)
ta["ratio_pt_car_high"] = (ta.pt_total_via_origin_min / (ta.car_min_high + CAR_TERMINAL_MIN)).round(2)

ta[["site", "commune_clean", "pt_total_via_origin_min", "car_door_mid_min", "ratio_pt_car_mid"]]

,site,commune_clean,pt_total_via_origin_min,car_door_mid_min,ratio_pt_car_mid
0,Sion,ARBAZ,40.2,30.0,1.34
1,Sion,ARDON,23.2,23.0,1.01
2,Sion,AYENT,46.0,25.0,1.84
3,Lausanne,BOTTENS,42.0,39.5,1.06
4,Sion,BRAMOIS,20.9,15.5,1.35
5,Lausanne,BUSSIGNY,20.6,24.0,0.86
6,Lausanne,CHAVANNES PRES RENENS,21.6,21.0,1.03
7,Lausanne,CHESEAUX SUR LAUSANNE,30.4,31.5,0.97
8,Sion,CONTHEY,21.1,16.0,1.32
9,Lausanne,CORCELLES LE JORAT,55.9,43.0,1.30


## A.6 Index and groups

- **Index:** mean of the 3 standardised indicators (z-scores): −(PT/car ratio), log(1 + departures 06:00–09:00), −(first arrival time). The higher the index, the better the PT access.
- **Groups:** terciles of the index, weighted by number of employees and **computed within each site**, so that every group contains employees from Lausanne (treated) and from Sion (control).

In [23]:
fa = pd.to_datetime(ta.first_arrival, format="%H:%M")
first_min = fa.dt.hour * 60 + fa.dt.minute
z = lambda s: (s - s.mean()) / s.std()
index = lambda ratio: (z(-ratio) + z(np.log1p(ta.n_departures_6_9)) + z(-first_min)) / 3
ta["transit_index"] = index(ta.ratio_pt_car_mid).round(3)       # higher = better PT access


def terciles(g):
    """3 groups of about 1/3 of the employees each, within one site."""
    g = g.sort_values("transit_index")
    position = (g.n_employees.cumsum() - g.n_employees / 2) / g.n_employees.sum()
    return pd.cut(position, [0, 1 / 3, 2 / 3, 1], labels=["1_low", "2_medium", "3_high"]).astype(str)


ta["transit_group"] = pd.concat([terciles(g) for _, g in ta.groupby("site")])

### Export: `transit_access.csv` and `table_A1.csv`

`transit_access.csv` is only rewritten when `RUN_API = True`, so that the submitted file is never overwritten by a re-run that simply reloads it.

In [24]:
ta["source"] = (f"Timetables: transport.opendata.ch (official SBB timetables), Thursday {REF_DATE}. "
                "Dwellings: RegBL via api3.geo.admin.ch. Communes: swisstopo. "
                "Car times: Google Maps, arrival 08:00 on a Thursday.")
ta["coding_notes"] = (
    "Selected stop = served stop (>=1 departure 06-09h) that minimises the average walk from all "
    "RegBL dwellings of the locality (straight line x1.3, 75 m/min, weighted by number of dwellings) + "
    "the fastest PT trip arriving 07:30-08:00; tie (<1 min) -> shortest walk. Candidates: "
    "stops close to the centre and to the centre of each quarter of the locality + main stop. "
    "Lausanne, Bramois, Salins: dwellings selected by the employees' postcode (1004, 1967, 1991). "
    "Destination: main station of the site. Services introduced after 2025 excluded "
    "(line 9 at Renens/Crissier, line 54, tram T1). Car: middle of the Google range + 5 min "
    "(parking + walking); ratio_pt_car_high = using the upper bound. "
    "transit_index = mean of the z-scores of (-PT/car ratio, log(1+departures 06-09h), -first "
    "arrival). transit_group = terciles of the index weighted by employees, computed within each site. "
    "Raw-data corrections: 'Savieze' -> SAVIESE; Salins postcode 1994 (= Aproz) -> 1991."
)

ta = ta.drop(columns="mean_distance_km", errors="ignore")
if RUN_API:
    ta.to_csv("transit_access.csv", index=False, encoding="utf-8-sig")

a1 = ta.sort_values(["site", "transit_index"], ascending=[True, False])[
    ["site", "commune_clean", "n_employees", "origin_stop", "pt_total_via_origin_min", "car_door_mid_min",
     "ratio_pt_car_mid", "n_departures_6_9", "first_arrival", "transit_index", "transit_group"]]
a1.to_csv("table_A1.csv", index=False, encoding="utf-8-sig")

a1

,site,commune_clean,n_employees,origin_stop,pt_total_via_origin_min,car_door_mid_min,ratio_pt_car_mid,n_departures_6_9,first_arrival,transit_index,transit_group
10,Lausanne,CRISSIER,45,"Crissier, Marcolet",23.3,27.0,0.86,49,04:59,1.287,3_high
18,Lausanne,LAUSANNE,138,"Lausanne, St-Paul",19.8,17.0,1.16,72,04:51,1.219,3_high
12,Lausanne,ECUBLENS,39,"Chavannes-R., Crochy",24.7,24.0,1.03,56,04:59,1.176,3_high
6,Lausanne,CHAVANNES PRES RENENS,20,"Chavannes-R., Concorde",21.6,21.0,1.03,47,04:59,1.106,3_high
27,Lausanne,RENENS,63,Renens VD,20.9,22.0,0.95,31,04:59,1.019,2_medium
23,Lausanne,MORGES,43,Morges,23.0,25.0,0.92,19,04:59,0.858,2_medium
25,Lausanne,PRILLY,43,Union-Prilly,20.7,17.5,1.18,28,04:51,0.826,2_medium
26,Lausanne,PULLY,49,Pully,16.8,16.5,1.02,25,05:11,0.765,2_medium
20,Lausanne,LUTRY,46,"Lutry, gare",17.6,22.0,0.80,12,05:11,0.696,2_medium
13,Lausanne,EPALINGES,29,"Epalinges, Croisettes",31.8,29.5,1.08,37,05:31,0.689,2_medium


## A.7 Checks

Distribution of employees across groups, and robustness to the choice of car time (upper bound of the Google range).

In [25]:
pd.set_option("display.width", 220)
print("\n", a1.to_string(index=False))
print("\nEmployees per group:\n", ta.pivot_table(index="site", columns="transit_group",
                                                 values="n_employees", aggfunc="sum"))
print("\nRank correlation between index (mid car time) and index (max car time):",
      round(ta.transit_index.corr(index(ta.ratio_pt_car_high), method="spearman"), 2))


     site         commune_clean  n_employees                    origin_stop  pt_total_via_origin_min  car_door_mid_min  ratio_pt_car_mid  n_departures_6_9 first_arrival  transit_index transit_group
Lausanne              CRISSIER           45             Crissier, Marcolet                     23.3              27.0              0.86                49         04:59          1.287        3_high
Lausanne              LAUSANNE          138              Lausanne, St-Paul                     19.8              17.0              1.16                72         04:51          1.219        3_high
Lausanne              ECUBLENS           39           Chavannes-R., Crochy                     24.7              24.0              1.03                56         04:59          1.176        3_high
Lausanne CHAVANNES PRES RENENS           20         Chavannes-R., Concorde                     21.6              21.0              1.03                47         04:59          1.106        3_high
Lausanne     

## A.8 Validation and limitations

The resulting ranking is plausible. Communes close to the sites and served by train or metro (Crissier, Lausanne, Ecublens, Chavannes-près-Renens; Ardon, Conthey) get the highest scores. Rural or mountain communes (Ropraz, Savigny, Servion; Hérémence, Saint-Martin, Ayent) get the lowest, with only 2 to 6 departures between 06:00 and 09:00 for the Valais communes. The index decreases with the reported distance to the site (rank correlation −0.51) and with PT travel time (−0.68). It is robust to the choice of car time: using the upper bound of the Google range, the rank correlation with the main index is 0.99.

Several limitations must be stressed.
1. The timetables are those of Thursday 10 December 2026 (2026 timetable), because the 2024–2025 timetables are no longer available. Services introduced after the survey were excluded.
2. Each commune is represented by a single stop and by all of its dwellings. In dispersed communes (Hérémence, Saint-Martin, Savièse, Savigny), the access walk exceeds 20 minutes and is probably overestimated, especially as the buildings register includes second homes.
3. Since the addresses of the sites are unknown, the destination is the main railway station.
4. Two inconsistencies in the raw data were corrected: the spelling of Savièse ("Savieze") and the postcode of Salins (1994, which belongs to Aproz, replaced by 1991).

## A.9 Merging the transit groups with the employee data

The transit index and the transit groups are defined at commune level (table `ta`). Parts B to E work at employee level, so we merge them onto the cleaned employee data `raw` using the harmonised commune name (`commune_clean`).

- The merge is many-to-one (many employee rows per commune, one row per commune in `ta`). `validate="m:1"` makes pandas raise an error if a commune appears twice in `ta`.
- The assertion checks that no employee is left without a transit group.
- The resulting table `df` (2,000 rows: 1,000 employees × 2 years) is the dataset used in all following parts. It contains the corrections made in A.1 (harmonised communes, corrected Salins postcode, completed incomes, car days capped at 5), and it is never re-read from `parking_data.csv`.
- The table below reports the number of employees in each site × transit group cell. It is the basis for the group sample sizes in Tables B2, C1 and E1.

In [26]:
df = raw.merge(ta[["commune_clean", "transit_index", "transit_group"]],
               on="commune_clean", how="left", validate="m:1")
assert df["transit_group"].notna().all()
assert len(df) == len(raw) == 2000
print(df.groupby(["site", "transit_group"])["employee_id"].nunique().unstack())

transit_group  1_low  2_medium  3_high
site                                  
Lausanne         285       273     242
Sion              66        68      66
